# pipeline

> Process a set of Euclid observations through the pipeline.

In [ ]:
# | default_exp euclid.pipeline

In [ ]:
# | exporti


import concurrent.futures
import logging
from functools import partial
from itertools import product
from typing import Callable, Optional, Any

from nicl.euclid.data_access import DataAccess
from nicl.euclid.utilities import default_data_path
from nicl.euclid.skyflat import (
    create_coarse_data,
    create_skyflats,
    group_obs_ids,
    write_skyflats,
)
from nicl.euclid.xarray import create_all_zarr_refs, read_all_zarr_refs
from nicl.euclid.persistence import correct_persistence
from nicl.euclid.combine import combine
from nicl.euclid.background_stats import measure

In [ ]:
# | hide
# Additional imports for documentation
from nbdev.showdoc import show_doc

In [ ]:
# | export

# Default parameters

RELEASE_NAME = "OTF"
ESAC_SERVER_URL = "https://easotf.esac.esa.int"
PROCESSING_VERSION = "v0.7"

SKYFLAT_N_PIX_NIR = 51
SKYFLAT_FILTER_SIZE_NIR = None
SKYFLAT_N_PIX_VIS = 149
SKYFLAT_FILTER_SIZE_VIS = None
SKYFLAT_HALF_WINDOW_NIR = 3
SKYFLAT_HALF_WINDOW_VIS = 5

PIXEL_SCALE = 0.3
MAX_WORKERS = 8
NIR_FILTERS = ["H", "J", "Y"]
FILTERS = NIR_FILTERS + ["I"]

In [ ]:
# | export


def possibly_concurrent(
    fn: Callable,  # Function to execute
    targets: list[Any],  # List of targets (e.g. obs_ids) to process
    *args,  # Positional arguments for fn
    executor: Optional[Callable] = None,  # Executor factory function
    **kwargs,  # Keyword arguments for fn
) -> list[Any]:
    """Run a function for each target, in parallel if executor is provided."""
    results = [None] * len(targets)
    if executor is not None:
        with executor() as e:
            # Submit all tasks
            futures = {
                e.submit(fn, target, *args, **kwargs): i
                for i, target in enumerate(targets)
            }

            # Process completed futures
            try:
                for future in concurrent.futures.as_completed(futures):
                    try:
                        result = future.result()
                        results[futures[future]] = result
                    except Exception as exc:
                        idx = futures[future]
                        logging.error(f"Task with target {targets[idx]} failed: {exc}")
            finally:
                # Ensure we cancel any remaining futures
                for f in futures:
                    f.cancel()
    else:
        # Sequential execution
        for i, target in enumerate(targets):
            try:
                result = fn(target, *args, **kwargs)
                results[i] = result
            except Exception as exc:
                logging.error(f"Task {i} failed: {exc}")
                raise
    return results


def get_required_obs_ids(target_obs_ids, available_obs_ids, half_window):
    """Get the list of obs_ids including neighboring observations required for skyflats."""
    target_obs_ids = list(target_obs_ids)
    group_for_obs_id = group_obs_ids(
        target_obs_ids, available_obs_ids, half_window=half_window
    )
    obs_ids = [obs_id for group in group_for_obs_id.values() for obs_id in group]
    obs_ids += target_obs_ids
    obs_ids = sorted(list(set(obs_ids)))
    return obs_ids


class Pipeline:
    """Process a set of Euclid observations through the pipeline.

     This class provides methods to process Euclid observations through various
     pipeline steps including data download, skyflat creation, persistence correction,
     stack creation, and background statistics calculation.

     The pipeline can be run as a context manager, which will ensure the parallel
     workers are cleaned up when the context is exited.
     ```python
     with Pipeline(target_obs_ids) as p:
         p.run_nir()
    ```
    """

    def __init__(
        self,
        target_obs_ids,
        release_name=RELEASE_NAME,
        release_folder_name=None,
        esac_server_url=ESAC_SERVER_URL,
        processing_version=PROCESSING_VERSION,
        skyflat_n_pix_nir=SKYFLAT_N_PIX_NIR,
        skyflat_filter_size_nir=SKYFLAT_FILTER_SIZE_NIR,
        skyflat_n_pix_vis=SKYFLAT_N_PIX_VIS,
        skyflat_filter_size_vis=SKYFLAT_FILTER_SIZE_VIS,
        skyflat_half_window_nir=SKYFLAT_HALF_WINDOW_NIR,
        skyflat_half_window_vis=SKYFLAT_HALF_WINDOW_VIS,
        pixel_scale=PIXEL_SCALE,
        max_workers=MAX_WORKERS,
        filters=FILTERS,
    ):
        self.target_obs_ids = list(
            dict.fromkeys(target_obs_ids)
        )  # remove duplicates, preserving order
        self.release_name = release_name
        if release_folder_name is None:
            self.release_folder_name = release_name
        else:
            self.release_folder_name = release_folder_name
        self.esac_server_url = esac_server_url
        self.processing_version = processing_version
        self.skyflat_n_pix_nir = skyflat_n_pix_nir
        self.skyflat_filter_size_nir = skyflat_filter_size_nir
        self.skyflat_n_pix_vis = skyflat_n_pix_vis
        self.skyflat_filter_size_vis = skyflat_filter_size_vis
        self.skyflat_half_window_nir = skyflat_half_window_nir
        self.skyflat_half_window_vis = skyflat_half_window_vis
        self.pixel_scale = pixel_scale
        self.filters = filters
        self.executor = None
        if max_workers > 1:
            self.executor = partial(
                concurrent.futures.ProcessPoolExecutor, max_workers=max_workers
            )
        self.logger = logging.getLogger(__name__)

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        pass

    def __del__(self):
        pass

    def _create_data_access(self):
        return DataAccess(
            esac_server_url=self.esac_server_url, release_name=self.release_name
        )

    def get_nir_data(self):
        """Download NIR data files."""
        self.logger.info("=== Downloading NIR Data ===")

        outpath = default_data_path(self.release_folder_name)
        self.logger.info(f"Saving data to {outpath}")

        da = self._create_data_access()
        available_obs_ids = da.find_all_observations()

        nir_obs_ids = get_required_obs_ids(
            self.target_obs_ids, available_obs_ids, self.skyflat_half_window_nir
        )
        self.logger.info(f"Downloading {len(nir_obs_ids)} NIR observations")
        self.logger.info(f"Required NIR obs_ids: {nir_obs_ids}")
        for obs_id in nir_obs_ids:
            self.logger.info(f"Downloading NIR files for {obs_id}")
            da.download_calibrated_files_for_observation(
                obs_id, instrument="NISP", outpath=outpath
            )
            da.download_sir_files_for_observation(obs_id, outpath=outpath)

    def get_vis_data(self):
        """Download VIS data files."""
        self.logger.info("=== Downloading VIS Data ===")

        outpath = default_data_path(self.release_folder_name)
        self.logger.info(f"Saving data to {outpath}")

        da = self._create_data_access()
        available_obs_ids = da.find_all_observations()

        vis_obs_ids = get_required_obs_ids(
            self.target_obs_ids, available_obs_ids, self.skyflat_half_window_vis
        )
        self.logger.info(f"Downloading {len(vis_obs_ids)} VIS observations")
        self.logger.info(f"Required VIS obs_ids: {vis_obs_ids}")
        for obs_id in vis_obs_ids:
            self.logger.info(f"Downloading VIS files for {obs_id}")
            da.download_calibrated_files_for_observation(
                obs_id, instrument="VIS", outpath=outpath
            )

    def _get_zarr_path(self, instrument):
        return default_data_path(
            f"{self.release_folder_name}_processed_{self.processing_version}",
            "zarr",
            instrument,
        )

    def _try_create_zarr_refs(self, obs_id, path, zarr_path):
        create_all_zarr_refs(path, zarr_path, obs_id_glob=f"{obs_id}")

    def create_zarr_refs(self, obs_ids, instrument):
        """Create zarr references for all the specified observations."""
        if instrument not in ["NIR", "VIS"]:
            raise ValueError(
                f"Invalid instrument: {instrument}. Must be 'NIR' or 'VIS'."
            )
        self.logger.info(f"Creating {instrument} zarr refs")
        if instrument == "VIS":
            data_instrument = "VIS_QUAD"
        else:
            data_instrument = instrument
        path = default_data_path(self.release_folder_name, data_instrument)
        possibly_concurrent(
            self._try_create_zarr_refs,
            obs_ids,
            path=path,
            zarr_path=self._get_zarr_path(instrument),
            executor=self.executor,
        )

    def _try_create_coarse_data(self, obs_id, instrument, zarr_path, n_pix):
        create_coarse_data(obs_id, zarr_path, instrument=instrument, n_pix=n_pix)

    def create_coarse_data(self, obs_ids, instrument):
        """Create coarse data for all the specified observations."""
        if instrument not in ["NIR", "VIS"]:
            raise ValueError(
                f"Invalid instrument: {instrument}. Must be 'NIR' or 'VIS'."
            )
        self.logger.info(f"Creating {instrument} coarse data")
        if instrument == "VIS":
            n_pix = self.skyflat_n_pix_vis
        else:
            n_pix = self.skyflat_n_pix_nir
        possibly_concurrent(
            self._try_create_coarse_data,
            obs_ids,
            instrument=instrument,
            zarr_path=self._get_zarr_path(instrument),
            n_pix=n_pix,
            executor=self.executor,
        )

    def _create_skyflats(self, instrument):
        """Create skyflats."""
        if instrument not in ["NIR", "VIS"]:
            raise ValueError(
                f"Invalid instrument: {instrument}. Must be 'NIR' or 'VIS'."
            )
        self.logger.info(f"=== Creating {instrument} Skyflats ===")
        zarr_path = self._get_zarr_path(instrument)
        outpath = default_data_path(
            f"{self.release_folder_name}_processed_{self.processing_version}",
            "skyflat",
            instrument,
        )

        da = self._create_data_access()
        available_obs_ids = da.find_all_observations()

        if instrument == "VIS":
            hw = self.skyflat_half_window_vis
            n_pix = self.skyflat_n_pix_vis
            filter_size = self.skyflat_filter_size_vis
            n_persistence = 0
        else:
            hw = self.skyflat_half_window_nir
            n_pix = self.skyflat_n_pix_nir
            filter_size = self.skyflat_filter_size_nir
            n_persistence = 1

        # Create zarr references and coarse data for all required observations
        obs_ids = get_required_obs_ids(self.target_obs_ids, available_obs_ids, hw)
        self.logger.info(
            f"Creating zarr refs for {len(obs_ids)} {instrument} observations"
        )
        self.create_zarr_refs(obs_ids, instrument)
        self.logger.info(
            f"Creating coarse data for {len(obs_ids)} {instrument} observations"
        )
        self.create_coarse_data(obs_ids, instrument)

        ds, wcs, _ = read_all_zarr_refs(zarr_path, obs_id_glob="*")
        available_obs_ids = sorted(ds.observation_id.values)
        group_for_obs_id = group_obs_ids(obs_ids, available_obs_ids, half_window=hw)

        for target_obs_id in self.target_obs_ids:
            for obs_id in range(
                target_obs_id - n_persistence, target_obs_id + n_persistence + 1
            ):
                if len(list(outpath.glob(f"flat-{obs_id}-*.fits"))) > 0:
                    self.logger.info(f"Skipping {obs_id} because it already exists")
                    continue
                if obs_id not in group_for_obs_id:
                    self.logger.warning(
                        f"zarr reference for {obs_id} is not available: skipping"
                    )
                    continue
                self.logger.info(f"Creating skyflats for obs_id: {obs_id}")
                self.logger.info(f"Using obs_ids: {group_for_obs_id[obs_id]}")
                flats = create_skyflats(
                    obs_id,
                    group_for_obs_id,
                    zarr_path,
                    instrument=instrument,
                    n_pix=n_pix,
                    filter_size=filter_size,
                )
                write_skyflats(obs_id, flats, outpath, coarse_factor=n_pix, wcs=wcs)

    def create_nir_skyflats(self):
        self._create_skyflats(instrument="NIR")

    def create_vis_skyflats(self):
        self._create_skyflats(instrument="VIS")

    def _try_correct_persistence(self, obs_id, path, processed_path, **kwargs):
        self.logger.info(f"Processing {obs_id}...")
        if kwargs.get("correct_persistence", True):
            label = "persistence"
        else:
            label = "no_persistence"
        if not kwargs.get("final_skyflat_correction", True):
            label += "_no_skyflat"
        if not kwargs.get("correct_banding", True):
            label += "_no_tartan"
        self.logger.info(f"Processing {obs_id} with {label}...")
        outpath = processed_path / f"{label}/NIR/{obs_id}/"
        skyflat_path = processed_path / "skyflat/NIR/"
        correct_persistence(
            obs_id, path, outpath=outpath, skyflat_path=skyflat_path, **kwargs
        )
        self.logger.info(f"Completed {obs_id}...")

    def do_persistence_correction(self, **kwargs):
        """Perform persistence correction."""
        self.logger.info("=== Performing Persistence Correction ===")
        path = default_data_path(self.release_folder_name)
        processed_path = default_data_path(
            f"{self.release_folder_name}_processed_{self.processing_version}"
        )
        self.logger.info(f"Processing {len(self.target_obs_ids)} observations:")
        self.logger.info(f"{self.target_obs_ids}")
        possibly_concurrent(
            self._try_correct_persistence,
            self.target_obs_ids,
            path=path,
            processed_path=processed_path,
            **kwargs,
            executor=self.executor,
        )

    def _try_combine(
        self, obs_id, in_dir, out_dir_parent, vis_skyflat_dir, filters, bkg_sub=True
    ):
        """Create stacks for obs_id if the output foler does not already exist."""
        out_dir = out_dir_parent / f"{obs_id}"
        if out_dir.exists():
            self.logger.info(f"Skipping {obs_id} because it already exists")
        else:
            self.logger.info(f"Creating stacks for obs_id: {obs_id}")
            try:
                kwargs = dict(
                    in_dir=in_dir,
                    out_dir=out_dir,
                    obs_ids=obs_id,
                    filters=filters,
                    bkg_sub=bkg_sub,
                    autodark_corr=True,
                    autodark_dir=vis_skyflat_dir,
                )
                if self.pixel_scale is not None:
                    kwargs["pixel_scale"] = self.pixel_scale
                combine(**kwargs)
            except Exception as e:
                self.logger.error(f"Error combining {obs_id}: {e}")
                raise

    def create_stacks(
        self,
        instrument,
        bkg_sub=True,
    ):
        """Create image stacks."""
        if instrument not in ["NIR", "VIS"]:
            raise ValueError(
                f"Invalid instrument: {instrument}. Must be 'NIR' or 'VIS'."
            )
        self.logger.info(f"=== Creating {instrument} Stacks ===")
        processed_path = default_data_path(
            f"{self.release_folder_name}_processed_{self.processing_version}"
        )
        if instrument == "NIR":
            in_dir = processed_path / "persistence" / "NIR"
            vis_skyflat_dir = None
        else:
            data_path = default_data_path(f"{self.release_folder_name}")
            in_dir = data_path / "VIS_QUAD"
            vis_skyflat_dir = processed_path / "skyflat" / "VIS"
        if bkg_sub:
            out_dir = processed_path / "stacked" / instrument
        else:
            out_dir = processed_path / "stacked_nobkg" / instrument
        filters = NIR_FILTERS if instrument == "NIR" else ["I"]
        filters = list(set(self.filters) & set(filters))
        possibly_concurrent(
            self._try_combine,
            self.target_obs_ids,
            in_dir=in_dir,
            out_dir_parent=out_dir,
            vis_skyflat_dir=vis_skyflat_dir,
            filters=filters,
            bkg_sub=bkg_sub,
            executor=self.executor,
        )

    def _try_background_stats(self, obs_id_and_filter, path, overwrite=False):
        """Measure background statistics and handle errors."""
        obs_id, filter = obs_id_and_filter
        if filter == "I":
            filename = f"EUC_VIS_SWL-STK-{obs_id}.fits"
        else:
            filename = f"EUC_NIR_W-STK_{filter}-{obs_id}.fits"
        obs_path = path / f"{obs_id}"
        if (obs_path / "background_stats" / filename).exists() and not overwrite:
            self.logger.info(f"Skipping {obs_id} {filter} because it already exists")
        else:
            self.logger.info(f"Measuring background stats for {obs_id} {filter}")
            try:
                measure(filename, obs_path)
            except Exception as e:
                self.logger.error(f"Error measuring {filename}: {e}")
                raise

    def calculate_background_stats(self, instrument, bkg_sub=True, overwrite=False):
        """Calculate background statistics."""
        if instrument not in ["NIR", "VIS"]:
            raise ValueError(
                f"Invalid instrument: {instrument}. Must be 'NIR' or 'VIS'."
            )
        self.logger.info(f"=== Calculating {instrument} Background Statistics ===")
        path = default_data_path(
            f"{self.release_folder_name}_processed_{self.processing_version}"
        )
        if bkg_sub:
            path = path / "stacked" / instrument
        else:
            path = path / "stacked_nobkg" / instrument
        filters = NIR_FILTERS if instrument == "NIR" else ["I"]
        filters = list(set(self.filters) & set(filters))
        obs_ids_and_filters = list(product(self.target_obs_ids, filters))
        possibly_concurrent(
            self._try_background_stats,
            obs_ids_and_filters,
            zp_mag="ZPAB",
            path=path,
            overwrite=overwrite,
            executor=self.executor,
        )

    def run_vis(self):
        """Run a default pipeline for the VIS data.

        You could run a custom version of this with a subset of the steps
        or with different parameters.

        One would typically not want to create stacks for every observation,
        but for a set of targets.

        The VIS and NIR pipelines are independent, so you could run one
        while the other is running.
        """
        self.logger.info("Starting processing pipeline for VIS data...")

        # Step 1: Download data
        self.get_vis_data()

        # Step 2: Create VIS skyflats
        self.create_vis_skyflats()

        # Step 3: Create stacks
        self.create_stacks(bkg_sub=False)

        # Step 4: Calculate background statistics
        self.calculate_background_stats(bkg_sub=False)

    def run_nir(self):
        """Run a default pipeline for the NIR data.

        You could run a custom version of this with a subset of the steps
        or with different parameters.

        One would typically not want to create stacks for every observation,
        but for a set of targets.

        The VIS and NIR pipelines are independent, so you could run one
        while the other is running.
        """
        self.logger.info("Starting processing pipeline for NIR data...")

        # Step 1: Download data
        self.get_nir_data()

        # Step 2: Create NIR skyflats
        self.create_nir_skyflats()

        # Step 3: Persistence correction
        self.do_persistence_correction()

        # Step 4: Create stacks
        self.create_stacks(bkg_sub=False)

        # Step 5: Calculate background statistics
        self.calculate_background_stats(bkg_sub=False)

In [ ]:
show_doc(Pipeline.get_nir_data)

---

[source](https://github.com/IntraClusterLight/nicl/blob/main/nicl/euclid/pipeline.py#L136){target="_blank" style="float:right; font-size:smaller"}

### Pipeline.get_nir_data

>      Pipeline.get_nir_data ()

*Step 1a: Download NIR data files.*

In [ ]:
show_doc(Pipeline.get_vis_data)

---

[source](https://github.com/IntraClusterLight/nicl/blob/main/nicl/euclid/pipeline.py#L158){target="_blank" style="float:right; font-size:smaller"}

### Pipeline.get_vis_data

>      Pipeline.get_vis_data ()

*Step 1b: Download VIS data files.*

In [ ]:
show_doc(Pipeline.create_nir_skyflats)

---

[source](https://github.com/IntraClusterLight/nicl/blob/main/nicl/euclid/pipeline.py#L270){target="_blank" style="float:right; font-size:smaller"}

### Pipeline.create_nir_skyflats

>      Pipeline.create_nir_skyflats ()

*Create NIR skyflats.*

In [ ]:
show_doc(Pipeline.do_persistence_correction)

---

[source](https://github.com/IntraClusterLight/nicl/blob/main/nicl/euclid/pipeline.py#L286){target="_blank" style="float:right; font-size:smaller"}

### Pipeline.do_persistence_correction

>      Pipeline.do_persistence_correction ()

*Step 3: Perform persistence correction.*

In [ ]:
show_doc(Pipeline.create_stacks)

---

[source](https://github.com/IntraClusterLight/nicl/blob/main/nicl/euclid/pipeline.py#L322){target="_blank" style="float:right; font-size:smaller"}

### Pipeline.create_stacks

>      Pipeline.create_stacks (bkg_sub=True)

*Step 4: Create image stacks.*

In [ ]:
show_doc(Pipeline.calculate_background_stats)

---

[source](https://github.com/IntraClusterLight/nicl/blob/main/nicl/euclid/pipeline.py#L357){target="_blank" style="float:right; font-size:smaller"}

### Pipeline.calculate_background_stats

>      Pipeline.calculate_background_stats (bkg_sub=True, overwrite=False)

*Step 5: Calculate background statistics.*

In [ ]:
show_doc(Pipeline.run_nir)

---

[source](https://github.com/IntraClusterLight/nicl/blob/main/nicl/euclid/pipeline.py#L402){target="_blank" style="float:right; font-size:smaller"}

### Pipeline.run_nir

>      Pipeline.run_nir ()

*Run a default pipeline for the NIR data.

You could run a custom version of this with a subset of the steps
or with different parameters.

One would typically not want to create stacks for every observation,
but for a set of targets.

The VIS and NIR pipelines are independent, so you could run one
while the other is running.*

In [ ]:
show_doc(Pipeline.run_vis)

---

[source](https://github.com/IntraClusterLight/nicl/blob/main/nicl/euclid/pipeline.py#L376){target="_blank" style="float:right; font-size:smaller"}

### Pipeline.run_vis

>      Pipeline.run_vis ()

*Run a default pipeline for the VIS data.

You could run a custom version of this with a subset of the steps
or with different parameters.

One would typically not want to create stacks for every observation,
but for a set of targets.

The VIS and NIR pipelines are independent, so you could run one
while the other is running.*